<tabla align="centro">
<td align="center"><a target="_blank" href="http://introtodeeplearning.com">
<img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
Visite el aprendizaje profundo del MIT</a></td>
<td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab1/solutions/PT_Part2_Music_Generation_Solution.ipynb">
<img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" />Ejecutar en Google Colab</a></td>
<td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab1/solutions/PT_Part2_Music_Generation_Solution.ipynb">
<img src="https://i.ibb.co/xfJbPmL/github.png" height="70px" style="padding-bottom:5px;"  />Ver código fuente en GitHub</a></td>
</tabla>

# Información de derechos de autor

In [ ]:
# Copyright 2026 MIT Introducción al aprendizaje profundo. Reservados todos los derechos.
#
# Licenciado bajo la Licencia MIT. No puede utilizar este archivo excepto en cumplimiento
# con la Licencia. Uso y/o modificación de este código fuera del MIT Introducción
# al Deep Learning debe hacer referencia a:
#
# © MIT Introducción al aprendizaje profundo
# http://intotodeeplearning.com
#

# Laboratorio 1: Introducción a PyTorch y Music Generation con RNN

# Parte 2: Generación musical con RNN

En esta parte del laboratorio, exploraremos la construcción de una red neuronal recurrente (RNN) para la generación de música utilizando PyTorch. Entrenaremos un modelo para aprender los patrones en partituras sin editar en [notación ABC] (https://en.wikipedia.org/wiki/ABC_notation) y luego usaremos este modelo para generar nueva música.

## 2.1 Dependencias
Primero, descarguemos el repositorio del curso, instalemos las dependencias e importemos los paquetes relevantes que necesitaremos para esta práctica de laboratorio.

Usaremos [Comet ML](https://www.comet.com/docs/v2/) para realizar un seguimiento de nuestro desarrollo de modelos y ejecuciones de capacitación. Primero, regístrese para obtener una cuenta de Comet [en este enlace](https://www.comet.com/signup?utm_source=mit_dl&utm_medium=partner&utm_content=github
) (puedes usar tu cuenta de Google o Github). Esto generará una clave API personal, que puede encontrar en la primera página "Comenzar con Comet", en la configuración de su cuenta o presionando el botón "?". en la esquina superior derecha y luego 'Guía de inicio rápido'. Ingrese esta clave API como la variable global `COMET_API_KEY`.

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
# TODO: ¡¡INGRESA TU CLAVE API AQUÍ!! instrucciones arriba
COMET_API_KEY = ""

# Importe PyTorch y otras bibliotecas relevantes
import torch
import torch.nn as nn
import torch.optim as optim

# Descargue e importe el paquete MIT Introducción al aprendizaje profundo
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# Importar todos los paquetes restantes
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1


# Comprobar que estamos usando una GPU, si no cambiar de tiempo de ejecución
# usando Runtime > Cambiar tipo de tiempo de ejecución > GPU
assert torch.cuda.is_available(), "Please enable GPU from runtime settings"
assert COMET_API_KEY != "", "Please insert your Comet API Key"



## 2.2 Conjunto de datos

![¡Bailemos!](http://33.media.tumblr.com/3d223954ad0a77f4e98a7b87136aa395/tumblr_nlct5lFVbF1qhu7oio1_500.gif)

Hemos recopilado un conjunto de datos de miles de canciones populares irlandesas, representadas en notación ABC. Descarguemos el conjunto de datos e inspeccionémoslo:


In [ ]:
# Descargar el conjunto de datos
songs = mdl.lab1.load_training_data()

# ¡Imprime una de las canciones para inspeccionarla con mayor detalle!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

Podemos convertir fácilmente una canción en notación ABC a una forma de onda de audio y reproducirla. Tenga paciencia hasta que se ejecute esta conversión, puede llevar algún tiempo.

In [ ]:
# Convierta la notación ABC a un archivo de audio y escúchelo
mdl.lab1.play_song(example_song)

Una cosa importante en la que pensar es que esta notación musical no solo contiene información sobre las notas que se tocan, sino que también hay metainformación como el título de la canción, la clave y el tempo. ¿Cómo afecta la cantidad de caracteres diferentes que están presentes en el archivo de texto a la complejidad del problema de aprendizaje? Esto será importante pronto, cuando generemos una representación numérica para los datos de texto.

In [ ]:
# Únase a nuestra lista de cadenas de canciones en una sola cadena que contenga todas las canciones
songs_joined = "\n\n".join(songs)

# Encuentra todos los caracteres únicos en la cadena unida
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## 2.3 Procesar el conjunto de datos para la tarea de aprendizaje

Demos un paso atrás y consideremos nuestra tarea de predicción. Estamos intentando entrenar un modelo RNN para aprender patrones en la música ABC y luego usar este modelo para generar (es decir, predecir) una nueva pieza musical basada en esta información aprendida.

Desglosando esto, lo que realmente le estamos preguntando al modelo es: dado un personaje, o una secuencia de personajes, ¿cuál es el siguiente personaje más probable? Entrenaremos el modelo para realizar esta tarea.

Para lograr esto, ingresaremos una secuencia de caracteres al modelo y entrenaremos el modelo para predecir la salida, es decir, el siguiente carácter en cada paso de tiempo. Los RNN mantienen un estado interno que depende de elementos vistos previamente, por lo que la información sobre todos los personajes vistos hasta un momento determinado se tendrá en cuenta para generar la predicción.

### Vectorizar el texto

Antes de comenzar a entrenar nuestro modelo RNN, necesitaremos crear una representación numérica de nuestro conjunto de datos basado en texto. Para hacer esto, generaremos dos tablas de búsqueda: una que asigna caracteres a números y una segunda que asigna números a caracteres. Recuerde que acabamos de identificar los caracteres únicos presentes en el texto.


In [ ]:
# ## Definir representación numérica del texto ###

# Cree una asignación de carácter a índice único.
# Por ejemplo, para obtener el índice del carácter "d",
# podemos evaluar `char2idx["d"]`.
char2idx = {u: i for i, u in enumerate(vocab)}

# Cree un mapeo de índices a caracteres. Esto es
# el inverso de char2idx y nos permite volver a convertir
# desde índice único hasta el carácter de nuestro vocabulario.
idx2char = np.array(vocab)

Esto nos da una representación entera para cada carácter. Observe que los caracteres únicos (es decir, nuestro vocabulario) en el texto se asignan como índices de 0 a `len(único)`. Echemos un vistazo a esta representación numérica de nuestro conjunto de datos:

In [ ]:
print('{')
for char, _ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
# ## Vectorizar la cadena de canciones ###

'''TODO: Write a function to convert the all songs string to a vectorized
    (i.e., numeric) representation. Use the appropriate mapping
    above to convert from vocab characters to the corresponding indices.

  NOTE: the output of the `vectorize_string` function
  should be a np.array with `N` elements, where `N` is
  the number of characters in the input string
'''
def vectorize_string(string):
  vectorized_output = np.array([char2idx[char] for char in string])
  return vectorized_output

# def vectorize_string(cadena):
# TODO

vectorized_songs = vectorize_string(songs_joined)

También podemos ver cómo se asigna la primera parte del texto a una representación entera:


In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# compruebe que vectorized_songs sea una matriz numerosa
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Crear ejemplos y objetivos de capacitación

Nuestro siguiente paso es dividir el texto en secuencias de ejemplo que usaremos durante el entrenamiento. Cada secuencia de entrada que introducimos en nuestro RNN contendrá caracteres `seq_length` del texto. También necesitaremos definir una secuencia objetivo para cada secuencia de entrada, que se utilizará en el entrenamiento del RNN para predecir el siguiente carácter. Para cada entrada, el destino correspondiente contendrá la misma longitud de texto, excepto que se desplazará un carácter hacia la derecha.

Para hacer esto, dividiremos el texto en fragmentos de `seq_length+1`. Supongamos que `seq_length` es 4 y nuestro texto es "Hola". Entonces, nuestra secuencia de entrada es "Infierno" y la secuencia de destino es "ello".

El método por lotes nos permitirá convertir este flujo de índices de caracteres en secuencias del tamaño deseado.


In [ ]:
# ## Definición de lotes para crear ejemplos de capacitación ###

def get_batch(vectorized_songs, seq_length, batch_size):
# la longitud de la cadena de canciones vectorizadas
    n = vectorized_songs.shape[0] - 1
# elija aleatoriamente los índices iniciales para los ejemplos en el lote de entrenamiento
    idx = np.random.choice(n - seq_length, batch_size)

    '''TODO: construct a list of input sequences for the training batch'''
    input_batch = [vectorized_songs[i: i + seq_length] for i in idx]

    '''TODO: construct a list of output sequences for the training batch'''
    output_batch = [vectorized_songs[i + 1: i + seq_length + 1] for i in idx]

# Convierta los lotes de entrada y salida en tensores
    x_batch = torch.tensor(input_batch, dtype=torch.long)
    y_batch = torch.tensor(output_batch, dtype=torch.long)

    return x_batch, y_batch

# ¡Realice algunas pruebas simples para asegurarse de que su función por lotes esté funcionando correctamente!
test_args = (vectorized_songs, 10, 2)
x_batch, y_batch = get_batch(*test_args)
assert x_batch.shape == (2, 10), "x_batch shape is incorrect"
assert y_batch.shape == (2, 10), "y_batch shape is incorrect"
print("Batch function works correctly!")


Para cada uno de estos vectores, cada índice se procesa en un único paso de tiempo. Entonces, para la entrada en el paso 0, el modelo recibe el índice del primer carácter de la secuencia e intenta predecir el índice del siguiente carácter. En el siguiente paso, hace lo mismo, pero el RNN considera la información del paso anterior, es decir, su estado actualizado, además de la entrada actual.

Podemos concretar esto observando cómo funciona esto en los primeros caracteres de nuestro texto:

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(x_batch[0], y_batch[0])):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx.item()])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx.item()])))


## 2.4 El modelo de red neuronal recurrente (RNN)

Now we're ready to define and train an RNN model on our ABC music dataset, and then use that trained model to generate a new song. We'll train our RNN using batches of song snippets from our dataset, which we generated in the previous section.

The model is based off the LSTM architecture, where we use a state vector to maintain information about the temporal relationships between consecutive characters. The final output of the LSTM is then fed into a fully connected linear [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer where we'll output a softmax over each character in the vocabulary, and then sample from this distribution to predict the next character.

As we introduced in the first portion of this lab, we'll be using PyTorch's [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) to define the model. Three components are used to define the model:

* [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html): This is the input layer, consisting of a trainable lookup table that maps the numbers of each character to a vector with `embedding_dim` dimensions.
* [`nn.LSTM`](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html): Our LSTM network, with size `hidden_size`.
* [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html): The output layer, with `vocab_size` outputs.

<img src="https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/lstm_unrolled-01-01.png" alt="Drawing"/>



<!--
Now we're ready to define and train a RNN model on our ABC music dataset, and then use that trained model to generate a new song. We'll train our RNN using batches of song snippets from our dataset, which we generated in the previous section.

The model is based off the LSTM architecture, where we use a state vector to maintain information about the temporal relationships between consecutive characters. The final output of the LSTM is then fed into a fully connected [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) layer where we'll output a softmax over each character in the vocabulary, and then sample from this distribution to predict the next character.

As we introduced in the first portion of this lab, we'll be using the Keras API, specifically, [`tf.keras.Sequential`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential), to define the model. Three layers are used to define the model:

* [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding): This is the input layer, consisting of a trainable lookup table that maps the numbers of each character to a vector with `embedding_dim` dimensions.
* [`tf.keras.layers.LSTM`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM): Our LSTM network, with size `units=rnn_units`.
* [`tf.keras.layers.Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense): The output layer, with `vocab_size` outputs.


<img src="https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/lstm_unrolled-01-01.png" alt="Drawing"/> -->

### Definir el modelo RNN

Definamos nuestro modelo como `nn.Module`. Complete los `TODO` para definir el modelo RNN.


In [ ]:
# ## Definición del modelo RNN ###

'''TODO: Add LSTM and Linear layers to define the RNN model using nn.Module'''
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size

# Definir cada una de las capas de la red.
# Capa 1: capa de incrustación para transformar índices en vectores densos
# de un tamaño de incrustación fijo
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

# Capa 2: LSTM con tamaño_oculto `tamaño_oculto`. nota: el número de capas por defecto es 1.
# TODO: Utilice el módulo nn.LSTM() de pytorch.
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
# self.lstm = nn.LSTM('''TODO''')

# Capa 3: capa lineal (completamente conectada) que transforma la salida de LSTM
# en el tamaño del vocabulario.
# TODO: Agrega la capa Lineal.
        self.fc = nn.Linear(hidden_size, vocab_size)
# self.fc = nn.Linear('''TODO''')

    def init_hidden(self, batch_size, device):
# Inicialice el estado oculto y el estado de la celda con ceros
        return (torch.zeros(1, batch_size, self.hidden_size).to(device),
                torch.zeros(1, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        x = self.embedding(x)

        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)

        out = self.fc(out)
        return out if not return_state else (out, state)



¡Ha llegado el momento! ¡Creamos una instancia del modelo!

In [ ]:
# ¡Crea una instancia del modelo! Cree un modelo simple con hiperparámetros predeterminados. Tú
# Tendrá la oportunidad de cambiarlos más tarde.
vocab_size = len(vocab)
embedding_dim = 256
hidden_size = 1024
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)

# imprimir un resumen del modelo
print(model)

### Pruebe el modelo RNN

Siempre es una buena idea ejecutar algunas comprobaciones sencillas en nuestro modelo para ver que se comporta como se esperaba.

Podemos verificar rápidamente las capas en el modelo, la forma de la salida de cada una de las capas, el tamaño del lote y la dimensionalidad de la salida. Tenga en cuenta que el modelo se puede ejecutar con entradas de cualquier longitud.

In [ ]:
# Pruebe el modelo con algunos datos de muestra.
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
x = x.to(device)
y = y.to(device)

pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

### Predicciones del modelo no entrenado

Echemos un vistazo a lo que predice nuestro modelo no entrenado.

Para obtener predicciones reales del modelo, tomamos muestras de la distribución de salida, que está definida por torch.softmax sobre nuestro vocabulario de caracteres. Esto nos dará índices de caracteres reales. Esto significa que estamos usando una [distribución categórica](https://en.wikipedia.org/wiki/Categorical_distribution) para tomar muestras de la predicción del ejemplo. Esto proporciona una predicción del siguiente carácter (específicamente su índice) en cada paso de tiempo. [`torch.multinomial`](https://pytorch.org/docs/stable/generated/torch.multinomial.html#torch.multinomial) muestra sobre una distribución categórica para generar predicciones.

Tenga en cuenta aquí que tomamos muestras de esta distribución de probabilidad, en lugar de simplemente tomar el "argmax", lo que puede hacer que el modelo se quede atascado en un bucle repetitivo.

Probemos este muestreo para el primer ejemplo del lote.

In [ ]:
sampled_indices = torch.multinomial(torch.softmax(pred[0], dim=-1), num_samples=1)
sampled_indices = sampled_indices.squeeze(-1).cpu().numpy()
sampled_indices

Ahora podemos decodificarlos para ver el texto predicho por el modelo no entrenado:

In [ ]:
print("Input: \n", repr("".join(idx2char[x[0].cpu()])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices])))

Como puede ver, el texto predicho por el modelo no entrenado no tiene sentido. ¿Cómo podemos hacerlo mejor? Bueno, ¡podemos entrenar la red!

## 2.5 Entrenamiento del modelo: operaciones de pérdida y entrenamiento

¡Ahora es el momento de entrenar al modelo!

En este punto, podemos pensar en nuestro próximo problema de predicción de personajes como un problema de clasificación estándar. Dado el estado anterior del RNN, así como la entrada en un paso de tiempo determinado, queremos predecir la clase del siguiente carácter, es decir, predecir realmente el siguiente carácter.

Para entrenar nuestro modelo en esta tarea de clasificación, podemos usar una forma de pérdida de "entropía cruzada" (es decir, pérdida de probabilidad logarítmica negativa). Específicamente, usaremos [`CrossEntropyLoss`] de PyTorch (https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), ya que combina la aplicación de un log-softmax ([`LogSoftmax`](https://pytorch.org/docs/stable/generated/torch.nn.LogSoftmax.html#torch.nn.LogSoftmax)) y probabilidad de registro negativa ([`NLLLoss`](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss) en una sola clase y acepta Objetivos enteros para tareas de clasificación categórica Querremos calcular la pérdida utilizando los objetivos verdaderos (las "etiquetas") y los objetivos previstos (los "logits").

Definamos una función para calcular la pérdida y luego usemos esa función para calcular la pérdida usando nuestras predicciones de ejemplo del modelo no entrenado.

In [ ]:
# ## Definiendo la función de pérdida ###

# '''TODO: definir la función Compute_loss para calcular y devolver la pérdida entre
# las verdaderas etiquetas y predicciones (logits). '''
cross_entropy = nn.CrossEntropyLoss() # instantiates the function
def compute_loss(labels, logits):
    """
    Inputs:
      labels: (batch_size, sequence_length)
      logits: (batch_size, sequence_length, vocab_size)

    Output:
      loss: scalar cross entropy loss over the batch and sequence length
    """

# Coloque las etiquetas en lotes de modo que la forma de las etiquetas sea (B * L,)
    batched_labels = labels.view(-1)

    ''' TODO: Batch the logits so that the shape of the logits should be (B * L, V) '''
    batched_logits = logits.view(-1, logits.size(-1))
# lotes_logits = """ TODO """ # TODO

    '''TODO: Compute the cross-entropy loss using the batched  next characters and predictions'''
    loss = cross_entropy(batched_logits, batched_labels)
# loss = """ TODO """ # TODO
    return loss

In [ ]:
# ## calcula la pérdida de las predicciones del modelo no entrenado anterior. ###
y.shape  # (batch_size, sequence_length)
pred.shape  # (batch_size, sequence_length, vocab_size)

'''TODO: compute the loss using the true next characters from the example batch
    and the predictions from the untrained model several cells above'''
example_batch_loss = compute_loss(y, pred)
# example_batch_loss = compute_loss('''TODO''', '''TODO''') # TODO

print(f"Prediction shape: {pred.shape} # (batch_size, sequence_length, vocab_size)")
print(f"scalar_loss:      {example_batch_loss.mean().item()}")

Comencemos definiendo algunos hiperparámetros para entrenar el modelo. Para empezar, hemos proporcionado algunos valores razonables para algunos de los parámetros. ¡Depende de usted usar lo que hemos aprendido en clase para ayudar a optimizar la selección de parámetros aquí!

In [ ]:
# ## Configuración y optimización de hiperparámetros ###

vocab_size = len(vocab)

# Parámetros del modelo:
params = dict(
  num_training_iterations = 3000,  # Increase this to train longer
  batch_size = 8,  # Experiment between 1 and 64
  seq_length = 100,  # Experiment between 50 and 500
  learning_rate = 5e-3,  # Experiment between 1e-5 and 1e-1
  embedding_dim = 256,
  hidden_size = 1024,  # Experiment between 1 and 2048
)

# Ubicación del punto de control:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

Una vez definidos nuestros hiperparámetros, podemos configurar el seguimiento de experimentos con Comet. [`Experiment`](https://www.comet.com/docs/v2/api-and-sdk/python-sdk/reference/Experiment/) son los objetos principales de Comet y nos permitirán realizar un seguimiento del entrenamiento y el desarrollo de modelos. Aquí hemos escrito una función breve para crear un nuevo experimento Comet. Tenga en cuenta que en esta configuración, cuando los hiperparámetros cambian, puede ejecutar la función `create_experiment()` para iniciar un nuevo experimento. Todos los experimentos definidos con el mismo `project_name` vivirán bajo ese proyecto en su interfaz Comet.



In [ ]:
# ## Crear un experimento Comet para realizar un seguimiento de nuestra ejecución de entrenamiento ###

def create_experiment():
# finalizar cualquier experimento previo
  if 'experiment' in locals():
    experiment.end()

# iniciar el experimento del cometa para el seguimiento
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name="6S191_Lab1_Part2")
# Registre nuestros hiperparámetros, definidos anteriormente, en el experimento.
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

Ahora estamos listos para definir nuestra operación de entrenamiento (el optimizador y la duración del entrenamiento) y usar esta función para entrenar el modelo. Experimentarás con la elección del optimizador y la duración durante la cual entrenas tus modelos, y verás cómo estos cambios afectan la salida de la red. Algunos optimizadores que le gustaría probar son [`Adam`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) y [`Adagrad`](https://pytorch.org/docs/stable/generated/torch.optim.Adagrad.html).

Primero, crearemos una instancia de un nuevo modelo y un optimizador, y los prepararemos para el entrenamiento. Luego, usaremos [`loss.backward()`](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html), habilitado por el método [autograd](https://pytorch.org/docs/stable/generated/torch.autograd.grad.html) de PyTorch, para realizar la retropropagación. Finalmente, para actualizar los parámetros del modelo según los gradientes calculados, daremos un paso con el optimizador, usando [`optimizer.step()`](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html).

También generaremos una copia impresa del progreso del modelo a través del entrenamiento, lo que nos ayudará a visualizar fácilmente si estamos minimizando o no la pérdida.

In [ ]:
# ## Definir optimizador y operación de entrenamiento ###

'''TODO: instantiate a new LSTMModel model for training using the hyperparameters
    created above.'''
model = LSTMModel(vocab_size, params["embedding_dim"], params["hidden_size"])
# modelo = LSTMModel('''TODO: argumentos''')

# Mover el modelo a la GPU
model.to(device)

'''TODO: instantiate an optimizer with its learning rate.
  Checkout the PyTorch website for a list of supported optimizers.
  https://pytorch.org/docs/stable/optim.html
  Try using the Adam optimizer to start.'''
optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])
# optimizer = # TODO

def train_step(x, y):
# Establecer el modo del modelo para entrenar.
  model.train()

# Cero gradientes para cada paso
  optimizer.zero_grad()

# pase hacia adelante
  '''TODO: feed the current input into the model and generate predictions'''
  y_hat = model(x) # TODO
# y_hat = model('''TODO''')

# Calcular la perdida
  '''TODO: compute the loss!'''
  loss = compute_loss(y, y_hat) # TODO
# loss = compute_loss('''TODO''', '''TODO''')

# pase hacia atrás
  '''TODO: complete the gradient computation and update step.
    Remember that in PyTorch there are two steps to the training loop:
    1. Backpropagate the loss
    2. Update the model parameters using the optimizer
  '''
  loss.backward() # TODO
  optimizer.step() # TODO

  return loss

# None
# ¡Empieza a entrenar!#
# None

history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
experiment = create_experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
for iter in tqdm(range(params["num_training_iterations"])):

# Tome un lote y propáguelo a través de la red.
    x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])

# Convierta matrices numerosas en tensores de PyTorch
    x_batch = torch.tensor(x_batch, dtype=torch.long).to(device)
    y_batch = torch.tensor(y_batch, dtype=torch.long).to(device)

# Da un paso de tren
    loss = train_step(x_batch, y_batch)

# Registre la pérdida en la interfaz Comet
    experiment.log_metric("loss", loss.item(), step=iter)

# Actualice la barra de progreso y visualícela en el cuaderno.
    history.append(loss.item())
    plotter.plot(history)

# Guardar punto de control del modelo
    if iter % 100 == 0:
        torch.save(model.state_dict(), checkpoint_prefix)

# Guarde el modelo entrenado final
torch.save(model.state_dict(), checkpoint_prefix)
experiment.flush()


## 2.6 Generar música usando el modelo RNN

¡Ahora podemos usar nuestro modelo RNN entrenado para generar algo de música! Al generar música, tendremos que alimentar al modelo con algún tipo de semilla para que comience (¡porque no puede predecir nada sin algo con lo que empezar!).

Una vez que tenemos una semilla generada, podemos predecir de forma iterativa cada carácter sucesivo (recuerde, estamos usando la representación ABC para nuestra música) usando nuestro RNN entrenado. Más específicamente, recuerde que nuestro RNN genera un "softmax" sobre posibles caracteres sucesivos. A modo de inferencia, tomamos muestras iterativamente de estas distribuciones y luego usamos nuestras muestras para codificar una canción generada en formato ABC.

Luego, ¡todo lo que tenemos que hacer es escribirlo en un archivo y escucharlo!

### El procedimiento de predicción

Ahora estamos listos para escribir el código para generar texto en formato de música ABC:

* Inicialice una cadena de inicio "semilla" y el estado RNN, y establezca la cantidad de caracteres que queremos generar.

* Utilice la cadena de inicio y el estado RNN para obtener la distribución de probabilidad sobre el siguiente carácter predicho.

* Muestra de distribución multinomial para calcular el índice del carácter predicho. Este carácter predicho se utiliza luego como la siguiente entrada al modelo.

* En cada paso de tiempo, el estado RNN actualizado se devuelve al modelo, de modo que ahora tenga más contexto para realizar la siguiente predicción. Después de predecir el siguiente carácter, los estados RNN actualizados se devuelven nuevamente al modelo, que es la forma en que aprende las dependencias de secuencia en los datos, a medida que obtiene más información de las predicciones anteriores.

![Inferencia LSTM](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/lstm_inference.png)

Complete y experimente con este bloque de código (¡así como con algunos de los aspectos de la definición y el entrenamiento de la red!) y vea cómo funciona el modelo. ¿Cómo se comparan las canciones generadas después del entrenamiento con una pequeña cantidad de épocas con las generadas después de un entrenamiento de mayor duración?

In [ ]:
# ## Predicción de una canción generada ###

def generate_text(model, start_string, generation_length=1000):
# Paso de evaluación (generar texto ABC utilizando el modelo RNN aprendido)

  '''TODO: convert the start string to numbers (vectorize)'''
  input_idx = [char2idx[s] for s in start_string] # TODO
# input_idx = ['''TODO''']
  input_idx = torch.tensor([input_idx], dtype=torch.long).to(device)

# Inicializar el estado oculto
  state = model.init_hidden(input_idx.size(0), device)

# Cadena vacía para almacenar nuestros resultados.
  text_generated = []
  tqdm._instances.clear()

  for i in tqdm(range(generation_length)):
    '''TODO: evaluate the inputs and generate the next character predictions'''
    predictions, state = model(input_idx, state, return_state=True)
# predicciones, estado_oculto = modelo('''TODO''', '''TODO''', return_state=True)

# Eliminar la dimensión del lote
    predictions = predictions.squeeze(0)

    '''TODO: use a multinomial distribution to sample over the probabilities'''
    input_idx = torch.multinomial(torch.softmax(predictions, dim=-1), num_samples=1)
# input_idx = torch.multinomial('''TODO''', dim=-1), num_samples=1)

    '''TODO: add the predicted character to the generated text!'''
# Sugerencia: considere en qué formato está la predicción frente a la salida
    text_generated.append(idx2char[input_idx].item()) # TODO
# text_generated.append('''TODO''')


  return (start_string + ''.join(text_generated))

In [ ]:
'''TODO: Use the model and the function defined above to generate ABC format text of length 1000!
    As you may notice, ABC files start with "X" - this may be a good start string.'''
generated_text = generate_text(model, start_string="X", generation_length=1000) # TODO
# generated_text = generate_text('''TODO''', '''TODO''', '''TODO''')

### ¡Reproduce la música generada!

¡Ahora podemos llamar a una función para convertir el texto en formato ABC en un archivo de audio y luego reproducirlo para ver nuestra música generada! Intente entrenar más tiempo si la canción resultante no es lo suficientemente larga, ¡o vuelva a generar la canción!

Guardaremos la canción en Comet; podrá encontrar sus canciones en las páginas `Audio` y `Assets & Artifacts` en la interfaz de Comet para el proyecto. Tenga en cuenta la documentación [`log_asset()`](https://www.comet.com/docs/v2/api-and-sdk/python-sdk/reference/Experiment/#experimentlog_asset), donde verá cómo especificar nombres de archivos y otros parámetros para guardar sus activos.

In [ ]:
# ## Reproducir canciones generadas ###

generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
# Sintetizar la forma de onda de una canción.
  waveform = mdl.lab1.play_song(song)

# Si es una canción válida (sintaxis correcta), ¡vamos a tocarla!
  if waveform:
    print("Generated song", i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

# guarde su canción en la interfaz de Comet; puede acceder a ella allí
    experiment.log_asset(wav_file_path)

In [ ]:
# cuando termine, finalice el experimento del cometa
experiment.end()

## 2.7 ¡Experimenta y **obtén premios por las mejores canciones**!

¡Felicitaciones por crear su primer modelo de secuencia en TensorFlow! Es un logro bastante grande y, con suerte, tendrás algunas melodías dulces para demostrarlo.

Considere cómo puede mejorar su modelo y qué parece ser más importante en términos de rendimiento. Aquí hay algunas ideas para comenzar:

* ¿Cómo afecta el número de épocas de entrenamiento al rendimiento?
* ¿Qué pasa si modifica o aumenta el conjunto de datos?
* ¿La elección de la cadena inicial afecta significativamente el resultado?

¡Intenta optimizar tu modelo y envía tu mejor canción! **Los participantes serán elegibles para recibir premios durante la oferta de enero de 2026. Para participar en el concurso, debe cargar lo siguiente en [este enlace de envío](https://www.dropbox.com/request/4hqfsOnLtX4jH1W3ynfp):**

* una grabación de tu canción;
* Cuaderno iPython con el código que usaste para generar la canción;
* una descripción y/o diagrama de la arquitectura y los hiperparámetros que utilizó. Si realizó modificaciones adicionales o interesantes en el código de la plantilla, inclúyalas en su descripción.

**Nombre su archivo en el siguiente formato: ``[Nombre]_[Apellido]_RNNMusic``, seguido del formato del archivo (.zip, .mp4, .ipynb, .pdf, etc.). Se prefieren los archivos ZIP de los tres componentes a los archivos individuales. Si envía archivos individuales, debe nombrar los archivos individuales de acuerdo con la nomenclatura anterior.**

¡También puedes enviarnos un tweet a [@MITDeepLearning](https://twitter.com/MITDeepLearning) con una copia de la canción (pero esto no te permitirá participar en el concurso)! Vea esta canción de ejemplo generada por un estudiante anterior (crédito a Ana Heart): <a href="https://twitter.com/AnaWhatever16/status/1263092914680410112?s=20">canción del 20 de mayo de 2020.</a>
<script async src="https://platform.twitter.com/widgets.js" charset="utf-8"></script>

¡Diviértete y feliz escuchando!

![¡Bailemos!](http://33.media.tumblr.com/3d223954ad0a77f4e98a7b87136aa395/tumblr_nlct5lFVbF1qhu7oio1_500.gif)
